# 1 Seq2Seq

## Machine Translation
### Seq2Seq 모델을 활용한 기계 번역
(영어를 불어로)

https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html

## 모듈 임포트

In [1]:
import os
import random
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

from tqdm.auto import tqdm

## 1. 데이터셋 준비

In [39]:
# <PAD>, <UNK>
SOS_token = 0
EOS_token = 1
batch_size = 64  # batch
epochs = 20  # epoch
latent_dim = 256  # context vactor 차원수
num_samples = 178000  # 학습데이터 검수
data_path = 'fra.txt' # 학습데이터 파일

In [40]:
len(lines)

178010

In [41]:
# 학습데이터 벡터화
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()
with open(data_path, 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')
for line in lines[: min(num_samples, len(lines) - 1)]:
    input_text, target_text, _ = line.split('\t') #데이터는 tab 으로 구분
    target_text = target_text
    input_texts.append(input_text)
    target_texts.append(target_text)
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)

input_characters = sorted(list(input_characters)) #입력 시퀀스 문자
target_characters = sorted(list(target_characters)) #출력 시퀀스 문자
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts]) #입력 시퀀스의 최대 길이
max_decoder_seq_length = max([len(txt) for txt in target_texts]) # 출력 시퀀스의 최대 길이
data_path = 'fra.txt' # 학습데이터 파일

In [42]:
num_encoder_tokens, num_decoder_tokens

(89, 113)

In [43]:
max_encoder_seq_length, max_decoder_seq_length

(174, 202)

In [44]:
input_token_index = dict(
    [('<SOS>', 0), ('<EOS>', 1)] + [(char, i+2) for i, char in enumerate(input_characters)]) #문자 : 인덱스 dictionary
target_token_index = dict(
    [('<SOS>', 0), ('<EOS>', 1)] + [(char, i+2) for i, char in enumerate(target_characters)]) #문자 : 인덱스 dictionary

n = len(input_texts)
encoder_input_data = np.zeros((n, max_encoder_seq_length+1), dtype=np.int32)
decoder_input_data = np.zeros((n, max_decoder_seq_length+1), dtype=np.int32)

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    data_encoded = []
    for t, char in enumerate(input_text):
        data_encoded.append(input_token_index[char])
    data_encoded.append(1) # EOS
    encoder_input_data[i, :len(data_encoded)] = np.array(data_encoded)

    # 타겟 데이터는 멀티클래스 분류 softmax 문제이기 때문에 60, 토큰 차원수로 벡터생성
    data_encoded = []
    for t, char in enumerate(target_text):
        data_encoded.append(target_token_index[char])
    data_encoded.append(1) # EOS
    decoder_input_data[i, :len(data_encoded)] = np.array(data_encoded)

In [45]:
encoder_input_data.shape

(178000, 175)

In [46]:
decoder_input_data.shape

(178000, 203)

In [47]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(encoder_input_data, decoder_input_data,test_size=0.2, shuffle=True)

In [48]:
# 데이터셋 만들기
class TextDataset(Dataset):
    def __init__(self, input_datas, target_datas):
        self.input_ids = torch.tensor(input_datas, dtype=torch.long)
        self.target_ids = torch.tensor(target_datas, dtype=torch.long)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

    def __len__(self):
        return len(self.input_ids)

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)

In [49]:
X_train[0]

array([46, 60, 57, 71, 57,  2, 71, 60, 67, 57, 71,  2, 55, 67, 71, 72,  2,
       72, 67, 67,  2, 65, 73, 55, 60, 12,  1,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0], dtype=int32)

In [50]:
train_dataset[0]

(tensor([46, 60, 57, 71, 57,  2, 71, 60, 67, 57, 71,  2, 55, 67, 71, 72,  2, 72,
         67, 67,  2, 65, 73, 55, 60, 12,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]),
 tensor([ 31,  59,  73,   2,  57,  62,  55,  75,  73,  73,  75,  72,  59,  73,
           2,  57,  69, 103,  74,  59,  68,  74,   2,  74,  72,  69,  70,   2,
          57,  62,  59,  72,  14,   1,   0,   0,  

## 2. 모델 정의

In [51]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size=256, dropout_p=0.1):
        super(EncoderRNN, self).__init__()

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden # output은 사용하지 않음

encoder = EncoderRNN(num_encoder_tokens+2)

In [52]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(max_decoder_seq_length+1):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        # decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.out(output)
        return output, hidden

decoder = DecoderRNN(256, num_decoder_tokens+2)

In [53]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
encoder.to(device)
decoder.to(device)

learning_rate = 1e-3

encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss().cuda()

## 3. 모델 학습

In [54]:
epochs = 10

history = {
    'loss': [],
    'val_loss': [],
    'val_acc': []
}

for epoch in range(1, epochs+1):
    # 학습
    encoder.train()
    decoder.train()
    train_loss = []
    for batch_idx, (input_tensor, target_tensor) in enumerate(tqdm(iter(train_loader))):
        input_tensor, target_tensor = input_tensor.to(device), target_tensor.to(device)

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        train_loss.append(loss.item())

    _train_loss = np.mean(train_loss)
    history['loss'].append(_train_loss)

    if epoch % 1 == 0:
        print(f'Epoch [{epoch}], Train Loss : [{_train_loss:.5f}]')


  0%|          | 0/2225 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4. 예측

In [17]:
input_idx2word = {v: k for k, v in input_token_index.items()}
target_idx2word = {v: k for k, v in target_token_index.items()}

In [18]:
input_tensor, target_tensor = test_dataset[0]
input_tensor, target_tensor = input_tensor.to(device), target_tensor.to(device)

In [22]:
input_tensor[0]

tensor(25, device='cuda:0')

In [23]:
# 입력값
for idx in input_tensor:
    print(input_idx2word[idx.item()], end='')

Do you see it?<EOS><SOS>

In [24]:
# 정답값
for idx in target_tensor:
    print(target_idx2word[idx.item()], end='')

	Est-ce que vous le voyez ?
<EOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS><SOS>

In [25]:
# 예측
encoder.eval()
decoder.eval()
with torch.no_grad():
    encoder_outputs, encoder_hidden = encoder(input_tensor.unsqueeze(0))
    decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

    loss = criterion(
        decoder_outputs.view(-1, decoder_outputs.size(-1)),
        target_tensor.view(-1)
    )
    print(loss)
    # val_loss.append(loss.item())

    _, topi = decoder_outputs.topk(1)
    decoded_ids = topi.squeeze()

    decoded_words = []
    for idx in decoded_ids:
        if idx.item() == EOS_token:
            decoded_words.append('<EOS>')
            break
        decoded_words.append(target_idx2word[idx.item()])

tensor(4.3543, device='cuda:0')


In [26]:
# \t == <SOS>
# \n == <EOS>

print(''.join(decoded_words))

	Tu peux sauver ?
<EOS>


In [ ]:
# Est-ce que vous le voyez ?

In [27]:
# 함수화
def predict_sentence(input_tensor):
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        encoder_outputs, encoder_hidden = encoder(input_tensor.unsqueeze(0))
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                # decoded_words.append('<EOS>')
                break
            decoded_words.append(target_idx2word[idx.item()])
    return decoded_words, loss


In [37]:
for seq_index in range(10):
    input_tensor, target_tensor = test_dataset[seq_index]
    input_tensor, target_tensor = input_tensor.to(device), target_tensor.to(device)

    print('------------')
    # 입력값
    print('input sentence: ', end=' ')
    indices = (input_tensor == 1).nonzero(as_tuple=True)[0]
    for idx in input_tensor[:indices]:
        print(input_idx2word[idx.item()], end='')

    # 정답값
    print('\ntarget sentence: ', end=' ')
    indices = (target_tensor == 1).nonzero(as_tuple=True)[0]
    for idx in target_tensor[:indices]:
        print(target_idx2word[idx.item()], end='')

    # 예측값
    print('\npredict sentence: ', end=' ')
    decoded_words, loss = predict_sentence(input_tensor)
    print(*decoded_words, sep='')

------------
input sentence:  Do you see it?
target sentence:  	Est-ce que vous le voyez ?

predict sentence:  	Tu peux sauver ?

------------
input sentence:  Don't shout.
target sentence:  	Ne crie pas.

predict sentence:  	Ne soyez pas fait pas !

------------
input sentence:  Am I first?
target sentence:  	Suis-je le premier ?

predict sentence:  	Suis-je en train de moi ?

------------
input sentence:  I did warn you.
target sentence:  	Je vous avais prévenus.

predict sentence:  	Je vous ai suisiste.

------------
input sentence:  I love hiking.
target sentence:  	J'adore les excursions.

predict sentence:  	J'aime les chiens.

------------
input sentence:  You all right?
target sentence:  	Tu vas bien ?

predict sentence:  	Tu es grand ?

------------
input sentence:  You look upset.
target sentence:  	Tu as l'air contrarié.

predict sentence:  	Tu es invitée.

------------
input sentence:  Are you single?
target sentence:  	Es-tu seul ?

predict sentence:  	Es-tu sourir ?

----